# 02 - Walk-forward backtest for the Standard LSTM

This notebook keeps evaluation chronological. Each prediction uses only prices available before the predicted next day. The reusable backtest implementation is in `src/backtest_engine.py`.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from backtest_engine import run_backtest


In [ ]:
price_path = PROJECT_ROOT / 'data' / 'raw' / 'stock_price.csv'
prices = pd.read_csv(price_path).iloc[2:].copy()
prices = prices.rename(columns={'Price': 'Date'})
prices['Date'] = pd.to_datetime(prices['Date'], errors='coerce')
prices['Close'] = pd.to_numeric(prices['Close'], errors='coerce')
prices = prices[['Date', 'Close']].dropna().sort_values('Date').reset_index(drop=True)
prices.head(), prices.tail(), prices.shape


## Walk-forward evaluation

The expanding history is shown explicitly below. The persistence prediction (`tomorrow = today`) is a transparent sanity-check baseline. The trained Standard LSTM metrics are loaded separately from `results/lstm_metrics.json`; this avoids retraining inside the notebook.

In [ ]:
lookback = 10
test_start = int(len(prices) * 0.8)
test_rows = prices.iloc[test_start - 1:].reset_index(drop=True)
current_prices = test_rows['Close'].to_numpy()[:-1]
actual_next_prices = test_rows['Close'].to_numpy()[1:]
walk_forward_predictions = current_prices.copy()
walk_forward_dates = test_rows['Date'].to_numpy()[1:]

walk_forward_result = run_backtest(
    dates=walk_forward_dates,
    prices=current_prices,
    predictions=walk_forward_predictions,
    actual_next_prices=actual_next_prices,
)
walk_forward_result['cumulative_return'], walk_forward_result['sharpe']


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(test_rows['Date'].iloc[1:], actual_next_prices, label='Actual next close')
plt.plot(test_rows['Date'].iloc[1:], walk_forward_predictions, '--', label='Walk-forward baseline')
plt.title('Chronological walk-forward sanity-check')
plt.xlabel('Date')
plt.ylabel('NDX close')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
metrics_path = PROJECT_ROOT / 'results' / 'lstm_metrics.json'
if metrics_path.exists():
    lstm_metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print(json.dumps(lstm_metrics, indent=2))
else:
    print('Run python src/train_lstm.py first to generate results/lstm_metrics.json.')
